# 02 — PPO with Pixel Observations
Train a PPO agent on Super Mario Bros using pixel-based input.

**Training strategy (based on Lior Shilon's article):**
- Phase 1: Train for 1,000,000 timesteps from scratch
- Phase 2: Load the best model and train for another 1,000,000 timesteps
- Checkpoint every 50,000 steps
- `learning_rate=1e-6`, `n_steps=512`, CnnPolicy

In [6]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [7]:
from src.wrappers import make_pixel_env
from src.agents import make_ppo_agent
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import PPOConfig

In [ ]:
# Configure and create agent (Phase 1 - first 1M steps)
config = PPOConfig(
    policy='CnnPolicy',
    learning_rate=1e-6,
    n_steps=512,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    total_timesteps=1_000_000,
    use_linear_lr_schedule=False,
)
model = make_ppo_agent(env, config=config, tensorboard_log='../logs/pixel_ppo')

In [ ]:
# Phase 1: Train for 1,000,000 timesteps
callback = CheckpointAndLogCallback(
    save_path='../models/pixel_ppo',
    save_freq=50_000,
)
model.learn(total_timesteps=config.total_timesteps, callback=callback)
model.save('../models/pixel_ppo/phase1_model')

In [ ]:
# Train
callback = CheckpointAndLogCallback(
    save_path='../models/pixel_ppo',
    save_freq=100_000,
)
model.learn(total_timesteps=config.total_timesteps, callback=callback)
model.save('../models/pixel_ppo/final_model')

Logging to ../logs/pixel_ppo/PPO_4
----------------------------
| time/              |     |
|    fps             | 116 |
|    iterations      | 1   |
|    time_elapsed    | 1   |
|    total_timesteps | 128 |
----------------------------
-------------------------------------------
| time/                   |               |
|    fps                  | 114           |
|    iterations           | 2             |
|    time_elapsed         | 2             |
|    total_timesteps      | 256           |
| train/                  |               |
|    approx_kl            | 2.4959445e-07 |
|    clip_fraction        | 0             |
|    clip_range           | 0.1           |
|    entropy_loss         | -1.95         |
|    explained_variance   | 0             |
|    learning_rate        | 0.00025       |
|    loss                 | 2.66e+03      |
|    n_updates            | 4             |
|    policy_gradient_loss | -6.92e-05     |
|    value_loss           | 5.33e+03      |
--------------

In [ ]:
# Phase 2: Load the phase 1 model and train for another 1,000,000 timesteps
from stable_baselines3 import PPO

model = PPO.load('../models/pixel_ppo/phase1_model')
model.set_env(env)

callback2 = CheckpointAndLogCallback(
    save_path='../models/pixel_ppo',
    save_freq=50_000,
)
model.learn(total_timesteps=1_000_000, callback=callback2)
model.save('../models/pixel_ppo/final_model')

In [ ]:
# Quick test
from src.utils.evaluation import evaluate_agent
results = evaluate_agent(model, env, n_episodes=10)
print(f"Mean reward: {results['mean_reward']:.1f} ± {results['std_reward']:.1f}")
print(f"Flag rate: {results['flag_rate']:.2%}")